```python
# Copyright (c) 2026 Louis Gevers
# SPDX-License-Identifier: BSD-3-Clause
```

# Legged Robots Project - Task 1: DRL on Flat Terrain

**Goal:** train a policy that makes the Go2 follow velocity commands in any direction on flat ground, then test one design choice of your own in a controlled experiment.

| Section | What you do |
| --- | --- |
| 1.1 | Build the environment, i.e. the MDP the policy learns in |
| 1.2 | Configure the policy network and the PPO algorithm |
| 1.3 | Train a baseline and monitor the training |
| 1.4 | Test the policy and collect evaluation data |
| 1.5 | Run a controlled experiment against your baseline |

This notebook **produces data**: trained policies, training logs and evaluation rollouts, all saved in `logs/`. 
It **requires a GPU for training**, evaluation and visualization can possibly be done on CPU but at a much slower rate.
You analyse the produced data in `logs/` in `02_flat_analysis.ipynb`, which is lightweight and runs entirely on CPU.
Take this into account when dividing work within your group!

**Time budget.** A training run takes roughly 10 to 20 minutes of shared GPU time. 
Check your environment in the viewer (1.1h) before training, and change one thing at a time between runs so you know what caused a difference.

If you are using gnoto, the GPUs might be busy during the practicals and **close to the deadline**.
Make sure to plan your training runs accordingly!

In [1]:
%load_ext autoreload
%autoreload 2

import math
import copy
import lr_project # Before mjlab import

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.rl import RslRlModelCfg, RslRlPpoAlgorithmCfg, RslRlOnPolicyRunnerCfg

from mjlab.managers.action_manager import ActionTermCfg
from mjlab.managers.command_manager import CommandTermCfg
from mjlab.managers.event_manager import EventTermCfg
from mjlab.managers.observation_manager import ObservationTermCfg, ObservationGroupCfg
from mjlab.managers.reward_manager import RewardTermCfg
from mjlab.managers.termination_manager import TerminationTermCfg

from lr_project import mdp, scene, play, train, evaluate, registry

## 1.1 Building the environment configuration

In mjlab, an environment is described by a configuration class, the `ManagerBasedRLEnvCfg`.
It is built from several parts, each defining one piece of the Markov Decision Process (MDP) which you will define in the sections below:


| MDP element | Config part | Section |
| --- | --- | --- |
| Task (what the robot should do) | commands | 1.1a |
| Observation $o_t$ | observations | 1.1b |
| Action $a_t$ | actions | 1.1c |
| Reward $r_t$ | rewards | 1.1d |
| End of an episode | terminations | 1.1e |
| Initial state distribution | events | 1.1f |
| Transition dynamics $P$ | MuJoCo physics (provided by `scene`) | 1.1g |

Each `make_*` function you will complete below returns a dictionary `{name: term config}`.
A term is a **function** (`func`) plus its optional **parameters** (`params`).
We provide you with ready-made functions in `src/lr_project/mdp/*.py` (skim through it!), and mjlab provides more in `mjlab.envs.mdp`.
You can also write your own functions in a cell and use it!

For more information on mjlab's syntax and available functions, consult the [official documentation](https://mujocolab.github.io/mjlab/main/index.html).


Every `make_*` function receives a `play` argument.
It is false for training (4096 parallel robots) and true for visualization and evaluation (1 robot, debug rendering of the commands).
Use it if a term should behave differently in the two cases.

**Advice:** it can be a bit daunting to define everything from scratch, where exactly do you start?
A helpful source of inspiration is taking a look at existing literature, for example the articles from the student presentations.
A starting point we recommend is the following paper by Rudin et al., which is a solid baseline for training quadruped locomotion policies using GPUs:

> Rudin, N., Hoeller, D., Reist, P. &amp; Hutter, M.. (2022). Learning to Walk in Minutes Using Massively Parallel Deep Reinforcement Learning. <i>Proceedings`of the 5th Conference on Robot Learning</i>, in <i>Proceedings of Machine Learning Research</i> 164:91-100 Available from https://proceedings.mlr.press/v164/rudin22a.html.

Note that this article also does challenging terrain, so implementing it exactly would be overkill for this policy that only needs to walk on flat ground.

### 1.1a Commands

The command defines the task: 
a desired base velocity $(v_x, v_y, \omega_z)$ in the robot's frame (forward and lateral speed in m/s, yaw rate in rad/s). 
Each robot samples a command uniformly from the ranges below and samples a new one every `resampling_time_range` seconds. 
The policy sees the command as an observation (1.1b) and is rewarded for tracking it (1.1d).

Choose ranges the Go2 can realistically reach. 
Things to think about:
- Why resample the command during an episode instead of once per episode?
- What happens if no robot is ever commanded to stand still (`rel_standing_envs`)?
- `rel_forward_envs` is the fraction of robots that only walk forward. Why could that help or hurt?

**Note:** there is a bug in mjlab's viewer when the command ranges are identical (it crashes), make sure to define them to realistic ranges before running the viewer.

In [ ]:
def make_commands(play: bool) -> dict[str, CommandTermCfg]:
    return {
        "joystick": mdp.UniformVelocityCommandCfg(
            # WHY: time for the policy to reach the target velocity before the next command
            # TRIED:
            # FUTURE:
            resampling_time_range=(5, 7), # TODO
            # WHY: keep some robots learning to stand still
            # TRIED:
            # FUTURE:
            rel_standing_envs=0.1, # TODO
            # WHY: forward walking is the most used skill, so slightly over-represented
            # TRIED:
            # FUTURE:
            rel_forward_envs=0.2, # TODO
            ranges=mdp.UniformVelocityCommandCfg.Ranges(
                # WHY: fast enough, but away from the Go2 speed limit (~2 m/s)
                # TRIED:
                # FUTURE: add negative values to learn walking backward
                lin_vel_x=(-1.5, 1.5), # TODO
                # WHY: small lateral variation around forward walking
                # TRIED:
                # FUTURE: widen: motion is too forward-centric now
                lin_vel_y=(-0.3, 0.3), # Rudin (-0.1, 0.1)
                # WHY: small turning variation around forward walking
                # TRIED:
                # FUTURE: widen: motion is too forward-centric now
                ang_vel_z=(-0.25, 0.25), # TODO
            ),
            entity_name="robot",
            debug_vis=play, # Do not render command arrows during training
        )
    }

### 1.1b Observations

Observations are what the networks see at every environment step. 
There are two groups:

- **actor**: the input of the policy. 
The policy will run on the robot, so think about what a real Go2 can measure or know before adding the observation.
- **critic**: the input of the value function. 
The critic is only used during training, so it may also receive *privileged* information that is easy to get in simulation but hard to measure on hardware (this is called an asymmetric actor-critic).

Terms are concatenated in the order you list them. 
Available functions are in `src/lr_project/mdp/observations.py`.

Things to think about:
What does the policy need to know at inference time?
Can the real robot make the measurements for the policy?
Can the critic infer the reward from its observations alone?

In [3]:
def make_observations(play: bool) -> dict[str, ObservationGroupCfg]:
    # VARIANT (1.5): the baseline is trained without observation noise; the variant adds
    # noise to the actor sensor terms, to compare a clean baseline with a noisy variant.
    actor = {
        # Example
        # WHY: the policy must know the target velocity
        # TRIED:
        # FUTURE:
        "command": ObservationTermCfg(
            func=mdp.generated_command,
            params={"command_name": "joystick"},
        ),
        # WHY: leg configuration, from the joint encoders
        # TRIED:
        # FUTURE:
        "joint_pos": ObservationTermCfg(
            func=mdp.joint_positions,
        ),
        # WHY: leg motion, from the joint encoders
        # TRIED:
        # FUTURE:
        "joint_vel": ObservationTermCfg(
            func=mdp.joint_velocities,
        ),
        # WHY: body rotation, measured directly by the IMU gyroscope
        # TRIED:
        # FUTURE:
        "base_angular_vel": ObservationTermCfg(
            func=mdp.base_ang_vel_b,
        ),
        # WHY: body tilt, from the IMU orientation estimate
        # TRIED:
        # FUTURE:
        "gravity": ObservationTermCfg(
            func=mdp.projected_gravity_b,
        ),
        # WHY: what the policy just commanded, helps smooth actions
        # TRIED:
        # FUTURE:
        "last_action": ObservationTermCfg(
            func=mdp.last_action,
        ),
        # WHY: short motion memory, to infer linear velocity without measuring it
        # TRIED:
        # FUTURE: ablation: single past action only
        "last_last_action": ObservationTermCfg(
            func=mdp.last_last_action,
        ),  
        # WHY: short motion memory, to infer linear velocity without measuring it
        # TRIED:
        # FUTURE: ablation: single past action only
        "last_last_last_action": ObservationTermCfg(
            func=mdp.last_last_last_action,
        ),
    }

    # ACCELERATION (not used):
    # joint acc: not measured on the Go2, double-differentiated encoders -> too noisy
    # base acc: real IMU accelerometer; gravity can be removed in software, but the remaining acc_x, acc_y
    #           are small, so even small impact spikes can hide them -> little usable information
    # eventually: add mdp.builtin_sensor with scene.BASE_IMU_LIN_ACC as an addition experiment

    critic = {
        **actor, # the critic sees everything the actor sees...
        # TODO: ...plus optional privileged terms (only available in simulation)
        # WHY: exact in sim but only estimated on the real robot: privileged, critic only
        # TRIED: in the actor (first run, 17:29): learned fine, but not reliable on hardware
        # FUTURE: ablation: actor with vs without it
        "base_lin_vel": ObservationTermCfg(
                    func=mdp.base_lin_vel_b,
                ),

    }

    return {
        "actor": ObservationGroupCfg(terms=actor),
        "critic": ObservationGroupCfg(terms=critic),
    }

### 1.1c Actions

The policy outputs 12 numbers, one per joint. 
With `JointPositionActionCfg`, each output is turned into a joint position target:

$$q^{target} = q^{default} + \text{scale} \cdot a_t \quad (\text{with } \texttt{use\_default\_offset=True})$$

A PD controller in the robot model then computes the motor torques that move the joint towards this target. 
The policy runs at 50 Hz, the PD controller and physics at 200 Hz.

Things to think about: Why predict offsets from the standing pose instead of absolute angles? What happens if `scale` is too small or too large?

While this is a common action space for quadrupeds, there are many other ways to translate policy actions into motor commands, and these can have a huge effect on your policy's performance!
For instance, one can use the policy outputs to modulate a CPG, as done previously in our lab:

> Bellegarda, G., & Ijspeert, A. (2022). CPG-RL: Learning central pattern generators for quadruped locomotion. IEEE Robotics and Automation Letters, 7(4), 12547-12554.

If interested in the topic, check out this paper for more action spaces and their effects on learning:

> Peng, X. B., & Van De Panne, M. (2017, July). Learning locomotion skills using deeprl: Does the choice of action space matter?. In Proceedings of the ACM SIGGRAPH/Eurographics Symposium on Computer Animation (pp. 1-13).

In [4]:
def make_actions(play: bool) -> dict[str, ActionTermCfg]:
    return {
        "joint_targets": mdp.JointPositionActionCfg(
            entity_name="robot", # Control the Go2 robot in the scene.
            actuator_names=(".*"), # Select all 12 leg motors: 3 per leg.
            # Imagine the policy controls a little virtual joystick for each joint.
            # True: a centred joystick asks for the normal standing pose.
            # Moving it asks for an adjustment away from that standing pose.
            # False: a centred joystick asks for zero joint angles, a different posture.
            # The network must learn which joystick positions produce a useful standing pose.
            # WHY: zero action = standing pose, so the robot is stable from the start and only learns corrections
            # TRIED:
            # FUTURE:
            use_default_offset=True,
            # Scale is the joystick sensitivity: how much the same action changes the target.
            # Small: gentle, small target changes. Large: bigger, potentially abrupt changes.
            # Zero: moving the joystick has no effect on the target.
            # Holding the joystick still keeps the same target; changes do not add up each step.
            # PD controllers move the actual joints towards these targets.
            # WHY: first try, kept small to get smooth motion
            # TRIED:
            # FUTURE:
            scale=0.25,
        ),
    }

### 1.1d Rewards

At every step the robot receives $r_t = \sum_i w_i \, f_i(s_t, a_t) \, \Delta t$: 
each term $f_i$ is multiplied by its weight $w_i$ (positive = reward, negative = penalty) and by the policy time step $\Delta t$.
Available functions are in `src/lr_project/mdp/rewards.py`:

- **Task**: `track_linear_velocity`, `track_angular_velocity` reward $\exp(-\text{error}^2 / \text{std}^2)$, so `std` sets how tolerant the tracking is (plot this function for better intuition!)
- **Penalties** (regularization): body velocities, joint velocities, accelerations, torques and power, action rate, contacts, ...
- **Shaping**: `feet_air_time` encourages stepping over jittering.

Terms that take a `sensor_name` need a sensor defined in the scene (the feet contact sensor is called `scene.FEET_CONTACT_SENSOR`).
Check `src/lr_project/scene.py` for more sensor names.

**How to tune:** 
start with the tracking terms and a few (minimal) penalties, then add or change **one term at a time**. 
Watch each term's curve (`Episode_Reward/<name>`) in TensorBoard (see 1.3a). 
If the robot finds an unexpected way to collect reward (e.g. vibrating in place), ask which term allows it. 

Keep a short log of what you tried!
You will explain your choices in notebook 02 and we will ask about them during the oral exam.

In [ ]:
def make_rewards(play: bool) -> dict[str, RewardTermCfg]:
    # Starting weights: tune using the robot's behaviour and each term's training curve.
    return {

        # Earn points for moving at the requested forward/sideways speed.
        # WHY: main task, reference weight (Rudin)
        # TRIED:
        # FUTURE:
        "track_linear_velocity": RewardTermCfg(
            func=mdp.track_linear_velocity,
            weight=1.0,
            params={"command_name": "joystick", "std": math.sqrt(0.125)},
        ),

        # Earn points for turning at the requested rate.
        # WHY: turning as important as linear
        # TRIED:
        # FUTURE: consider 0.5 as Rudin
        "track_angular_velocity": RewardTermCfg(
            func=mdp.track_angular_velocity,
            weight=1,
            params={"command_name": "joystick", "std": math.sqrt(0.125)},
        ),

        # Earn a small bonus for lifting feet while movement is requested.
        # WHY: small shaping term, kept well below tracking (it pays every step)
        #      +1 per foot in the air for 0.05-0.5 s, only if |v_xy|+|w_z| > 0.5:
        #      real steps; no reward for vibrating (<0.05 s), for long swings (>0.5 s), or when standing
        # TRIED:
        # FUTURE:
        #"feet_air_time": RewardTermCfg(
        #    func=mdp.feet_air_time,
        #    weight=0.1,
        #    params={
        #        "sensor_name": scene.FEET_CONTACT_SENSOR,
        #        "command_name": "joystick",
        #    },
        #),

        # Lose points for bouncing up/down along the body's z-axis.
        # WHY: Rudin; strong, vertical bouncing is wasted motion
        # TRIED:
        # FUTURE:
        #"body_linear_velocity_penalty": RewardTermCfg(
        #    func=mdp.body_linear_velocity_penalty,
        #    weight=-2.0,
        #),

        # Lose points for rocking sideways or pitching back/forth; yaw turning is allowed.
        # WHY: Rudin; light, some roll/pitch is natural in a gait
        # TRIED:
        # FUTURE:
        #"body_angular_velocity_penalty": RewardTermCfg(
        #    func=mdp.body_angular_velocity_penalty,
        #    weight=-0.05,
        #),

        # Lose points for leaning, even if the body is not rocking.
        # WHY: our starting value: moderate, keeps the body level without blocking motion
        # TRIED:
        # FUTURE:
        #"body_tilt": RewardTermCfg(
        #    func=mdp.flat_orientation_l2,
        #    weight=-0.5,
        #),

        # Lose points for jerking the virtual joysticks between policy steps.
        # WHY: Rudin; smoothness without blocking fast leg motion
        # TRIED:
        # FUTURE:
        #"action_rate": RewardTermCfg(
        #    func=mdp.action_rate_l2,
        #    weight=-0.01,
        #),

        # Lose points for unnecessarily large motor efforts.
        # WHY: tiny: torque^2 is large (tens of Nm), so a small weight is enough
        # TRIED:
        # FUTURE: consider -1e-5 as Rudin
        #"joint_torques": RewardTermCfg(
        #    func=mdp.joint_torques_l2,
        #    weight=-0.0001,
        #),

        # Lose points for leg contacts.
        # WHY: our starting value: discourage thigh/calf contacts; light, not a fall
        # TRIED:
        # FUTURE:
        #"leg_contact_penalty": RewardTermCfg(
        #    func=mdp.contacts_penalty,
        #    weight=-0.1,
        #    params={"sensor_name": scene.LEG_CONTACT_SENSOR},
        #),

        # Lose points for body contacts.
        # WHY: almost no effect: base contact already terminates, so it fires only on that last step
        # TRIED:
        # FUTURE: consider removing
        #"base_contact_penalty": RewardTermCfg(
        #    func=mdp.contacts_penalty,
        #    weight=-0.3,
        #    params={"sensor_name": scene.BASE_CONTACT_SENSOR},
        #)
    }

### 1.1e Terminations

An episode ends when any termination term returns `True` for a robot, and that robot is reset (1.1f).
There are two kinds:

- **Time-outs** (`time_out=True`): 
the episode reached its time limit. 
This is not a failure: PPO keeps estimating the value of the last state, as if the robot could continue (why is this necessary?)
- **Failures** (`time_out=False`): 
the robot fell. 
The robot loses all future rewards, which acts as an implicit penalty, and no more samples are wasted on a robot lying on the ground.

Things to think about: 
What should count as a fall? 
What happens if the failure condition is too strict or too loose?

In [6]:
def make_terminations(play: bool) -> dict[str, TerminationTermCfg]:
    return {
        # Example
        # WHY: 20 s limit is not a failure: PPO keeps valuing the last state
        # TRIED:
        # FUTURE:
        "time_out": TerminationTermCfg(
            func=mdp.time_out,
            time_out=True, # TODO: is this a failure or not?
        ),
        # TODO: add at least one failure termination that detects a fall

        # The body has tipped more than 60 degrees away from upright.
        # WHY: past 60 deg the Go2 cannot recover; looser wastes samples, stricter cuts normal motion
        # TRIED:
        # FUTURE:
        "bad_orientation": TerminationTermCfg(
            func=mdp.bad_orientation,
            params={"limit_angle": math.radians(60)},
            time_out=False,
        ),

        # The base has dropped below 15 cm: treat this as a collapse.
        # WHY: well below standing height (~0.3 m): the robot has collapsed
        # TRIED:
        # FUTURE:
        "low_height": TerminationTermCfg(
            func=mdp.root_height_below_minimum,
            params={"minimum_height": 0.15},
            time_out=False,
        ),

        # End the attempt if the robot's body touches the ground.
        # WHY: body on the ground = fall
        # TRIED:
        # FUTURE:
        "base_contact": TerminationTermCfg(
            func=mdp.illegal_contact,
            params={"sensor_name": scene.BASE_CONTACT_SENSOR},
            time_out=False,
        ),

    }

### 1.1f Events

Events are functions that modify the simulation at given moments, set by `mode`:

- `"startup"`: once, when the environment is created;
- `"reset"`: every time a robot starts a new episode;
- `"interval"`: periodically during an episode (e.g. pushes).

Here, reset events define the **initial state distribution**: where and how robots start. 
`pose_range` gives ranges for the base position (m, relative to its spawn point) and orientation (rad). 
`reset_joints_by_offset` can also randomize the initial joint angles.

Things to think about:
Why do the initial poses need to be randomized?
Is it as important on flat terrain than on a challenging terrain scenario?

In [7]:
def make_events(play: bool) -> dict[str, EventTermCfg]:
    return {
        "reset_base": EventTermCfg(
            func=mdp.reset_root_pose_uniform,
            mode="reset",
            params={
                "pose_range": {
                    # Start near the robot's assigned starting location.
                    # WHY:
                    # TRIED:
                    # FUTURE:
                    "x": (-0.25, 0.25),
                    # WHY:
                    # TRIED:
                    # FUTURE:
                    "y": (-0.25, 0.25),

                    # Start slightly above the normal standing height.
                    # WHY:
                    # TRIED:
                    # FUTURE:
                    "z": (0.04, 0.06),

                    # Give the body a small initial lean, about +/-3 degrees.
                    # WHY:
                    # TRIED:
                    # FUTURE:
                    "roll": (-0.05, 0.05),
                    # WHY:
                    # TRIED:
                    # FUTURE:
                    "pitch": (-0.05, 0.05),

                    # Start facing any direction.
                    # WHY:
                    # TRIED:
                    # FUTURE:
                    "yaw": (-math.pi, math.pi),
                },
            },
        ),

        "reset_joints": EventTermCfg(
            func=mdp.reset_joints_by_offset,
            mode="reset",
            params={
                # Start close to the normal standing leg posture.
                # WHY:
                # TRIED:
                # FUTURE:
                "position_range": (-0.05, 0.05),

                # Start with the joints at rest.
                # WHY:
                # TRIED:
                # FUTURE:
                "velocity_range": (0.0, 0.0),
            },
        ),
    }

### 1.1g Assembly

All parts come together here, with the scene (robot, flat terrain, feet contact sensor) and the simulator settings. 
The physics runs at 200 Hz (`timestep_s=0.005`) and the policy acts every 4 physics steps (`decimation=4`), i.e. at 50 Hz. 
An episode lasts 20 s = 1000 policy steps. 
You normally do not need to change this cell.

In [8]:
def make_env_cfg(play: bool = False, seed: int = 42) -> ManagerBasedRlEnvCfg:
    return ManagerBasedRlEnvCfg(
        scene=scene.make_scene_cfg(
            num_envs=1 if play else 4096,
            terrain="flat",
            robot_entity_name="robot",
        ),
        sim=scene.make_sim_cfg(timestep_s=0.005),
        actions=make_actions(play),
        commands=make_commands(play),
        events=make_events(play),
        observations=make_observations(play),
        rewards=make_rewards(play),
        terminations=make_terminations(play),
        decimation=4,
        episode_length_s=20.0,
        seed=seed,
    )

### 1.1h Visualize your environment

Run your environment with a `"random"` or `"zero"` policy before training. 
Check that robots start where you expect, that the command arrows look right, and that a fallen robot is reset. 
Problems found here cost seconds while problems found after training cost a training run!

Stop the viewer with the **interrupt** button (■) before running the next cell.

In [9]:
env_cfg = make_env_cfg(play=True)
play.interactive(env_cfg, policy="random") # "random" or "zero"

Setting seed: 42
Warp 1.18.0 initialized:
   CUDA Toolkit 13.4, Driver 13.2
   Devices:
     "cpu"      : "CPU"
     "cuda:0"   : "NVIDIA GeForce RTX 5060 Laptop GPU" (8 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/carolina-araujo/.cache/warp/1.18.0
Module mujoco_warp._src.smooth b9cc505 load on device 'cuda:0' took 3.07 ms  (cached)
Module _nxn_broadphase__locals__kernel_3a0c7ab9 3a0c7ab load on device 'cuda:0' took 0.98 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_9914afce 9914afc load on device 'cuda:0' took 1.99 ms  (cached)
Module mujoco_warp._src.constraint 9435a53 load on device 'cuda:0' took 1.07 ms  (cached)
Module _friction_dof__locals__kernel_35a89db9 35a89db load on device 'cuda:0' took 1.09 ms  (cached)
Module _limit_slide_hinge__locals__kernel_6395e840 6395e84 load on device 'cuda:0' took 0.87 ms  (cached)
Module _efc_contact_init__locals__kernel_81f4db55 81f4db5 load on device 'cuda:0' took 0.67 ms  (cached)
Module _efc_contact_j

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

URL to interactive simulation: http://localhost:8080

Ctrl+C received. Shutting down viewer...


(viser) Server stopped

Viewer closed!


/home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/.venv/lib/python3.12/site-packages/mjlab/viewer/viser/reward_bar_panel.py:86: UserWarning: Attempted to remove an already removed GuiHtmlHandle.
  self._html_handle.remove()


## 1.2 Building the training configuration

The policy is trained with **PPO** (Proximal Policy Optimization), an actor-critic algorithm from the `rsl_rl` library. 
Each iteration has two phases:

1. **Collect:** all 4096 robots run the current policy for `num_steps_per_env` steps.
2. **Update:** the actor and critic networks are improved using the collected data.

The values below are a reasonable starting point. 
**Change them only with a reason** (i.e., after having a working baseline), and one at a time.

### 1.2a Actor model

The actor is the policy: a neural network (MLP) mapping the actor observations to the mean of a Gaussian distribution over actions. 
During training, actions are sampled from this distribution to explore: `init_std` is the initial exploration noise, and the standard deviation is learned. 
`obs_normalization` rescales each observation using running statistics (why is this needed?).

In [10]:
def make_actor_cfg() -> RslRlModelCfg:
    return RslRlModelCfg(
        hidden_dims=(128, 128, 128),
        activation="elu",
        obs_normalization=True,
        distribution_cfg={
            "class_name": "GaussianDistribution",
            "init_std": 1.0,
            "std_type": "scalar",
        },
    )

### 1.2b Critic model

The critic estimates the value $V(s)$: 
the expected discounted sum of future rewards from a state. 
PPO uses it to judge whether an action turned out better or worse than expected (the *advantage*). 
It takes the critic observations as input and is not used once training is finished.

In [11]:
def make_critic_cfg() -> RslRlModelCfg:
    return RslRlModelCfg(
        hidden_dims=(128, 128, 128),
        activation="elu",
        obs_normalization=True,
    )

### 1.2c PPO hyperparameters

| Parameter | Meaning |
| --- | --- |
| `gamma` | Discount factor $\gamma$: how much future rewards count. $\gamma = 0.995$ at 50 Hz looks about 4 s ahead. |
| `lam` | GAE $\lambda$: trade-off between bias and variance of the advantage estimate. |
| `clip_param` | Limits how much the policy can change in one update (stabilizes training). |
| `entropy_coef` | Bonus for keeping the action distribution wide (exploration). |
| `num_learning_epochs`, `num_mini_batches` | How many passes over the collected data, split into how many batches. |
| `learning_rate`, `schedule`, `desired_kl` | Learning rate for fitting the neural networks, `"adaptive"` lowers it if the policy changes more than `desired_kl` per update. |
| `value_loss_coef`, `use_clipped_value_loss` | Weight and clipping of the critic loss. |
| `max_grad_norm` | Gradient clipping, for stability. |


In [12]:
def make_ppo_cfg() -> RslRlPpoAlgorithmCfg:
    return RslRlPpoAlgorithmCfg(
        value_loss_coef=1.0,
        use_clipped_value_loss=True,
        clip_param=0.2,
        entropy_coef=0.01,
        num_learning_epochs=5,
        num_mini_batches=4,
        learning_rate=1.0e-3,
        schedule="adaptive",
        gamma=0.995,
        lam=0.97,
        desired_kl=0.01,
        max_grad_norm=1.0,
    )

### 1.2d Assembly

`make_rl_cfg` combines the actor-criticm models with the PPO hyperparameters, and sets the size of training: 
each iteration collects 4096 robots × 24 steps ≈ 98k transitions, for `max_iterations` iterations. 
A checkpoint is saved every `save_interval` iterations and at the end. 
The `seed` makes a run reproducible: train the same configuration with different seeds to see how much results vary by chance!

**Do not change the `max_iterations` argument!**
For flat terrain locomotion, your policies should comfortably converge within 500 iterations (and keeps training within acceptable time frames for shared GPU usage).
If it doesn't, tune your MDP further.

In [13]:
def make_rl_cfg(seed: int = 42) -> RslRlOnPolicyRunnerCfg:
    return RslRlOnPolicyRunnerCfg(
        seed=seed,
        num_steps_per_env=24,
        max_iterations=500, # DO NOT INCREASE
        save_interval=500,
        actor=make_actor_cfg(),
        critic=make_critic_cfg(),
        algorithm=make_ppo_cfg(),
    )

## 1.3 Training

In this section you will run and monitor your training using the previously defined MDP and RL configurations.

### 1.3a Monitoring your training with Tensorboard

Start TensorBoard **before** training, then open the printed link in a new tab. 
Refresh the page to update the curves while training runs.
Useful curves to monitor and compare across trainings:

- `Train/mean_reward` and `Train/mean_episode_length` (1000 steps = no robot falls within 20 s);
- `Episode_Reward/<term>`: the contribution of each reward term, to see what the policy is optimizing;
- `Loss/*` and the policy's action standard deviation, to see whether learning is stable and exploration shrinks.

In [14]:
train.tensorboard()

Tensorboard server running! Monitor your training here: http://localhost:37751/


### 1.3b Running your training

Training saves everything in `logs/<experiment>/<date>_<variant>/`. 
Running the cell again starts a **new** run, previous runs are kept.
Make sure to back up your `logs` as producing the data and checkpoints is computationally expensive.

- Use `experiment="flat"` for all runs of this task, and `variant` to name what the run tests (initially your baseline)
- To train the same configuration with another seed, pass `seed=...` to both `make_env_cfg` and `make_rl_cfg` and keep the same variant name.

In [15]:
train.train(
    env_cfg=make_env_cfg(),
    rl_cfg=make_rl_cfg(),
    experiment="flat",
    variant="baseline",
)

[train] run dir: /home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-14_12-02-24_baseline
[train] device: cuda:0, envs: 4096, seed: 42
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 4096   |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms

PosixPath('/home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-14_12-02-24_baseline')

### 1.3c Loading your latest training directory

The registry finds runs by experiment and variant, so this cell also works after a kernel restart: 
you never need to retrain to continue and can skip the training cell. 
`runs(...)` returns the runs sorted by creation time, and `[-1]` takes the latest.

In [16]:
run = registry.runs(experiment="flat", variant="baseline")[-1]
print(f"Found checkpoint at: {run.checkpoint}")

Found checkpoint at: /home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-14_12-02-24_baseline/model_499.pt


## 1.4 Evaluation

### 1.4a Testing your policy 

Watch your trained policy in the viewer. 
Does it follow the commands? 
Does the gait look natural, or does it shuffle, hop or vibrate? 
Is it stable when the command changes? 
What you see here guides your next reward changes!
Stop the viewer (■) when done.

In [17]:
play.interactive(
    env_cfg=make_env_cfg(play=True),
    rl_cfg=make_rl_cfg(),
    policy=run.checkpoint,
)

Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 8      |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms.
+-------------------------------------------+
|            Active Command Terms           |
+-------+----------+------------------------+
| Index | Name     |          

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

URL to interactive simulation: http://localhost:8080


opening handshake failed
Traceback (most recent call last):
  File "/home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/.venv/lib/python3.12/site-packages/websockets/http11.py", line 138, in parse
    request_line = yield from parse_line(read_line)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/.venv/lib/python3.12/site-packages/websockets/http11.py", line 320, in parse_line
    line = yield from read_line(MAX_LINE_LENGTH)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/.venv/lib/python3.12/site-packages/websockets/streams.py", line 46, in read_line
    raise EOFError(f"stream ends after {p} bytes, before end of line")
EOFError: stream ends after 0 bytes, before end of line

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/carolina-araujo/Docu

(viser) Connection opened (0, 1 total), 249 persistent messages

(viser) Connection closed (0, 0 total)


Ctrl+C received. Shutting down viewer...


(viser) Server stopped

Viewer closed!


/home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/.venv/lib/python3.12/site-packages/mjlab/viewer/viser/reward_bar_panel.py:86: UserWarning: Attempted to remove an already removed GuiHtmlHandle.
  self._html_handle.remove()


### 1.4b Collecting rollout data for analysis

`collect_rollouts` runs the policy on a fixed grid of commands, with `robots_per_command` robots per command for `duration_s` seconds, and records everything (base motion, joint states, torques, foot contacts and ground reaction forces). 
The data is saved as a `Rollout` file in the run directory, under `eval/<name>_<checkpoint>.npz`, and is what you analyse in notebook 02.
For more information on what exactly gets recorded, check out `src/lr_project/rollout.py`.

- Running the cell again **loads** the saved file instead of simulating again, unless the settings changed (or `overwrite=True`).
- Use the **same commands and name** for every run you want to compare, so the comparison is fair.

In [19]:
# Fixed commands to evaluate: all combinations of the values below
# NOTE: keep this grid fixed once the baseline is final: changing it means re-collecting the baseline rollouts
commands = evaluate.command_grid(
    # TODO: what commands are you interested in evaluating?


    # WHY: covers the training range (min, middle, max)
    # TRIED:
    # FUTURE: add 0 to test standing still (trained with rel_standing_envs)
    vx=(0.0, 0.5, 1.0, 1.5), # m/s
    # WHY: training range plus 0, so pure forward walking is included
    # TRIED:
    # FUTURE:
    vy=(-0.1, 0, 0.1), # m/s
    # WHY: training range plus 0, so pure forward walking is included
    # TRIED:
    # FUTURE:
    wz=(-0.25, 0, 0.25), # rad/s
)

# Name of the saved evaluation, reuse it for every run you compare
eval_name = "command_grid"

evaluate.collect_rollouts(
    env_cfg=make_env_cfg(play=True),
    rl_cfg=make_rl_cfg(),
    checkpoint=run.checkpoint,
    commands=commands,
    name=eval_name,
    robots_per_command=10, # robots per unique command combination
    duration_s=10.0,
    seed=42,
)

print(f"Saved evaluation metrics at: {run.eval_file(eval_name)}")

[evaluate] command_grid_model_499.npz is outdated or has different settings, re-running
[evaluate] creating 36 commands x 10 robots = 360 environments
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 360    |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms.
+----------------

100%|██████████| 500/500 [00:02<00:00, 190.21it/s]


Saved evaluation metrics at: /home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-14_12-02-24_baseline/eval/command_grid_model_499.npz


## 1.5 Controlled experiment

⚠️ Start this section only **after** having a working baseline you are satisfied with and will **not tune further anymore**!

Now test one design choice of your own that is different from your baseline (e.g., a potential improvement).
A controlled experiment changes **exactly one factor** relative to your baseline and keeps everything else fixed: training budget, evaluation commands and analysis. 
Possible types of experiments:

| Type | What changes | Example |
| --- | --- | --- |
| Ablation | Remove a component | Remove an observation term or a penalty |
| Addition | Add a component | Add a reward term or observation noise |
| Substitution | Replace a design choice | A different action space or command distribution |
| Sensitivity | One parameter at a few levels | Action scale, a reward weight, `gamma` |

Results vary between seeds, so one run per configuration can mislead! 
If time allows, train the baseline and the variant with several seeds each.

### 1.5a Experiment design

**[ANSWER]** Fill this in **before** training your new variant:

- **Question:** what do you want to find out?
- **Type:** ablation, addition, substitution or sensitivity.
- **Change:** exactly what differs from the baseline.
- **Held fixed:** what stays the same.
- **Possible confounders:** what else might change as a side effect (e.g. a new action scale also changes the size of the action-rate penalty), and how you deal with it.
- **Prediction:** which metrics you expect to change, in which direction, and why.

> *Answer here, ~1-2 sentences per question*

### 1.5b Building variant config

Start from a copy of your baseline config and change exactly one thing. 
Change **either** the env config or rl config (not both) depending on your question.
Generally, env config changes lead to more interesting questions.

In [ ]:
VARIANT_NAME = "variant" # TODO: short descriptive name, e.g. "no_action_rate"

def make_variant_env_cfg(play: bool = False, seed: int = 42) -> ManagerBasedRlEnvCfg:
    base_cfg = make_env_cfg(play=play, seed=seed)

    variant_cfg = copy.deepcopy(base_cfg)

    # TODO: if env cfg variant, change exactly one thing, for example:
    # variant_cfg.rewards.pop("action_rate")                        # remove a reward term
    # variant_cfg.rewards["track_linear_velocity"].weight = 2.0     # change a weight
    # variant_cfg.actions["joint_targets"].scale = 0.5              # change the action scale

    return variant_cfg

def make_variant_rl_cfg(seed: int = 42) -> RslRlOnPolicyRunnerCfg:
    base_cfg = make_rl_cfg(seed=seed)

    variant_cfg = copy.deepcopy(base_cfg)

    # TODO: if rl cfg variant, change exactly one thing, for example:
    # variant_cfg.algorithm.gamma = 0.99

    return variant_cfg

### 1.5c Visualize your variant

Check that the variant builds and behaves as intended before spending GPU time on it:

In [ ]:
env_cfg = make_variant_env_cfg(play=True)
play.interactive(env_cfg, policy="random") # "random" or "zero"

### 1.5d Train your variant

Same experiment as the baseline, with your variant name. 
Then load the variant's latest run.

In [ ]:
train.train(
    env_cfg=make_variant_env_cfg(),
    rl_cfg=make_variant_rl_cfg(),
    experiment="task1_flat",
    variant=VARIANT_NAME,
)

In [ ]:
variant_run = registry.runs(experiment="flat", variant="baseline")[-1]
print(f"Found checkpoint at: {variant_run.checkpoint}")

### 1.5e Test your variant policy

Compare qualitatively with what you saw for the baseline in 1.4a. 
The quantitative comparison will take place in notebook 02.

In [ ]:
play.interactive(
    env_cfg=make_variant_env_cfg(play=True),
    rl_cfg=make_variant_rl_cfg(),
    policy=variant_run.checkpoint,
)

### 1.5f Collect variant rollouts

Use exactly the same `commands` and `eval_name` as for the baseline (1.4b), so that notebook 02 compares them equally.

In [ ]:
evaluate.collect_rollouts(
    env_cfg=make_variant_env_cfg(play=True),
    rl_cfg=make_variant_rl_cfg(),
    checkpoint=variant_run.checkpoint,
    commands=commands,
    name=eval_name,
    robots_per_command=10,
    duration_s=10.0,
    seed=42,
)

print(f"Saved evaluation metrics at: {run.eval_file(eval_name)}")

## Next step

You now have a baseline and a variant, each with training logs and evaluation rollouts.
Continue in `02_flat_analysis.ipynb`.